# Portfolio Theory with Matrix Algebra — Julia

Julia twin of `portfolio_theory_python.ipynb`: the same pipeline — log-return
matrix, correlation/covariance, positive-semidefinite check, portfolio moments,
Sharpe-ratio optimization with train/test validation — in Julia with
`Optim.jl`.

**Data:** this notebook reads `prices_long.csv`, the shared cache written
by the Python notebook (Yahoo Finance, 2020 → today, 11 GICS sectors). Run the
Python notebook once first, or point `CACHE` at your own long-format file with
columns `Date, Ticker, Close, Sector`.

Merged from `part1_Matrix_algebra_julia.ipynb` + `part2_Matrix_algebra_julia.ipynb`.

## 0. Setup

In [ ]:
using CSV, DataFrames, LinearAlgebra, Statistics, Dates, PlotlyJS, Optim

CACHE = "prices_long.csv"   # written by portfolio_theory_python.ipynb

## 1. Data — shared cache from the Python notebook

Long format (`Date, Ticker, Close, Sector`), same shape the original
`load_stocks` produced from the sector CSVs.

In [ ]:
long = CSV.read(CACHE, DataFrame)
long.Date = DateTime.(long.Date)
sort!(long, [:Ticker, :Date])
println("rows: ", nrow(long), "  tickers: ", length(unique(long.Ticker)))
println("range: ", minimum(long.Date), " -> ", maximum(long.Date))
first(long, 5)

## 2. Log returns and the return matrix

$R_{t,i} = \log(P_{t,i}/P_{t-1,i})$ — tickers as columns, dates as rows.

In [ ]:
# Calculate log returns
long = transform(groupby(long, :Ticker),
    :Close => (x -> [missing; log.(x[2:end] ./ x[1:end-1])]) => :LogReturn)

# Pivot: Tickers as columns, Dates as rows, LogReturn as values
ret_df = unstack(long, :Date, :Ticker, :LogReturn)
sort!(ret_df, :Date)

# same cleaning as the Python notebook: keep tickers with >500 observations,
# then drop rows with any gap so matrix products never see `missing`
all_tickers = names(ret_df)[2:end]   # skip :Date
keep = [t for t in all_tickers if count(!ismissing, ret_df[!, t]) > 500]
ret_df = select(ret_df, [:Date; Symbol.(keep)])
ret_df = dropmissing(ret_df)

numeric_cols = names(ret_df)[2:end]   # skip :Date
ret_matrix = Matrix{Float64}(ret_df[:, numeric_cols])
println("return matrix: ", size(ret_matrix), " (dates x tickers)")

## 3. Correlation, covariance, and the PSD check

A correlation matrix must be positive semidefinite — all eigenvalues ≥ 0.

In [ ]:
corr_matrix = cor(ret_matrix, dims=1)
eigenvalues = eigvals(Symmetric(corr_matrix))
is_psd = all(eigenvalues .>= -1e-8)

println("Is the correlation matrix PSD? ", is_psd)
println("Smallest eigenvalue: ", minimum(eigenvalues))
println("Largest eigenvalue:  ", maximum(eigenvalues))

## 4. Portfolio metrics — the equal-weighted baseline

$\mu_p = w^T\mu$, $\sigma_p^2 = w^T\Sigma w$,
$\mathrm{Sharpe}_p = \mu_p/\sigma_p$.

In [ ]:
n_assets = size(ret_matrix, 2)
W = ones(n_assets) / n_assets          # equal weights
mu = vec(mean(ret_matrix, dims=1))
sd = vec(std(ret_matrix, dims=1))
cov_matrix = cov(ret_matrix, dims=1)

port_mean = dot(W, mu)
port_vol = sqrt(W' * cov_matrix * W)
println("assets: ", n_assets)
println("equal-weight portfolio: mean=", round(port_mean, digits=6),
        " vol=", round(port_vol, digits=6),
        " Sharpe=", round(port_mean / port_vol, digits=4))

println("\nTop 5 assets by individual Sharpe (mean/std):")
for (t, s) in zip(numeric_cols, mu ./ sd) |> collect |> x -> sort(x, by=last, rev=true) |> x -> x[1:5]
    println("  $t: ", round(s, digits=4))
end

## 5. Cumulative log returns

In [ ]:
cum = cumsum(ret_matrix, dims=1)
traces = [scatter(x=ret_df.Date, y=cum[:, j], mode="lines",
                  name=numeric_cols[j], line=attr(width=1))
          for j in 1:n_assets]
plot(traces, Layout(title="Cumulative Log Returns (all tickers)",
                    xaxis_title="Date", yaxis_title="Cumulative log return",
                    width=1000, height=600, showlegend=false))

## 6. Sharpe-ratio optimization

$\max_w \; w^T\mu / \sqrt{w^T\Sigma w}$ s.t. $w \ge 0$.
Two solvers: **Fminbox(LBFGS)** (box constraints, renormalized objective)
and **L-BFGS** on a normalized objective (the $\sum w = 1$ budget baked in via scale invariance).

In [ ]:
function sharpe_pf(W, returns_matrix)
    cov_mat = cov(returns_matrix, dims=1)
    mu = vec(mean(returns_matrix, dims=1))
    pf_risk = sqrt(W' * cov_mat * W)
    return dot(W, mu) / pf_risk
end
println("Equal-weight Sharpe: ", round(sharpe_pf(W, ret_matrix), digits=4))

# --- Fminbox(LBFGS): box constraints, renormalize inside the objective ---
mu_v = vec(mean(ret_matrix, dims=1))
cov_v = cov(ret_matrix, dims=1)

function neg_sharpe_box(w)
    wn = w / sum(w)
    risk = sqrt(wn' * cov_v * wn)
    return risk == 0 ? Inf : -(dot(wn, mu_v) / risk)
end

res_box = Optim.optimize(neg_sharpe_box, zeros(n_assets), ones(n_assets),
                   ones(n_assets) / n_assets, Fminbox(LBFGS()),
                   Optim.Options(iterations=300))
w_box = res_box.minimizer / sum(res_box.minimizer)
println("Fminbox(LBFGS) Sharpe: ", round(-neg_sharpe_box(w_box), digits=4),
        "  converged: ", Optim.converged(res_box))

# --- Box-constrained L-BFGS with the budget baked in ---
# Sharpe(w) is scale-invariant, so maximizing Sharpe(w / sum(w)) over the
# box 0 <= w <= 1 is equivalent to the budget-constrained problem.
function neg_sharpe_nrm(w)
    wn = w / sum(w)
    r = ret_matrix * wn
    s = std(r)
    return s == 0 ? Inf : -mean(r) / s
end
res_nrm = Optim.optimize(neg_sharpe_nrm, zeros(n_assets), ones(n_assets),
                         ones(n_assets) / n_assets, Fminbox(LBFGS()),
                         Optim.Options(iterations=500))
w_nrm = res_nrm.minimizer / sum(res_nrm.minimizer)
println("L-BFGS (budget baked in) Sharpe: ", round(-neg_sharpe_nrm(w_nrm), digits=4),
        "  converged: ", Optim.converged(res_nrm))

println("\nTop holdings (Fminbox):")
for (t, wt) in zip(numeric_cols, w_box)
    wt > 0.001 && println("  $t: ", round(wt, digits=4))
end

## 7. Certified global optimum — JuMP + Gurobi (AMPL course license)

Max-Sharpe is non-convex, so L-BFGS above can only promise a local optimum.
But Sharpe is scale-invariant: with $y = w / (\mu^\top w)$,
$$\max_{w \ge 0, \; \sum w = 1} \frac{\mu^\top w}{\sqrt{w^\top \Sigma w}}
\;\Longleftrightarrow\;
\min_{y \ge 0} \; y^\top \Sigma y \quad \text{s.t.} \; \mu^\top y = 1,$$
and $w^\star = y^\star / \sum y^\star$. That is a convex QP, so Gurobi
certifies the **global** optimum. The cell below drives the AMPL-bundled
Gurobi driver from Julia via JuMP + AmplNLWriter — the same course license
used on the Python side. Set `AMPL_BIN`/`GUROBI_BIN` to your amplpy install
(the driver finds `ampl.lic` through `PATH`).

In [ ]:
# --- AMPL course license: point Julia at your amplpy installation ---
# export AMPL_BIN="<venv>/lib/python3.12/site-packages/ampl_module_base/bin"      # holds ampl + ampl.lic
# export GUROBI_BIN="<venv>/lib/python3.12/site-packages/ampl_module_gurobi/bin/gurobi"
AMPL_BIN   = get(ENV, "AMPL_BIN", "")
GUROBI_BIN = get(ENV, "GUROBI_BIN", "")
if isempty(AMPL_BIN) || isempty(GUROBI_BIN)
    error("Set AMPL_BIN and GUROBI_BIN env vars to your amplpy install (see comments above).")
end
ENV["PATH"] = AMPL_BIN * ":" * ENV["PATH"]   # so the driver finds ampl.lic

using JuMP, AmplNLWriter
model = Model(() -> AmplNLWriter.Optimizer(GUROBI_BIN))
@variable(model, yg[1:n_assets] >= 0)
@objective(model, Min, sum(cov_matrix[i,j]*yg[i]*yg[j] for i in 1:n_assets, j in 1:n_assets))
@constraint(model, sum(mu[i]*yg[i] for i in 1:n_assets) == 1)
optimize!(model)
yv = value.(yg)
w_g = yv / sum(yv)
r_g = ret_matrix * w_g
println("JuMP/Gurobi status: ", termination_status(model))
println("Global max Sharpe:  ", round(mean(r_g)/std(r_g), digits=4),
        "  sum(w): ", round(sum(w_g), digits=6),
        "  max(w): ", round(maximum(w_g), digits=4))

## 8. Train/test validation

60% train / 40% test. Optimize on the most volatile assets in-sample, score
the frozen weights out-of-sample against equal weight.

In [ ]:
split = Int(floor(size(ret_matrix, 1) * 0.6))
train = ret_matrix[1:split, :]
test = ret_matrix[split+1:end, :]
println("train: ", size(train), "  test: ", size(test))

# most volatile assets, in-sample
vols = vec(std(train, dims=1))
top_idx = sortperm(vols, rev=true)[1:min(550, length(vols))]
train_reduced = train[:, top_idx]
tickers_reduced = numeric_cols[top_idx]

mu_tr = vec(mean(train_reduced, dims=1))
cov_tr = cov(train_reduced, dims=1)
n_tr = length(mu_tr)

function neg_sharpe_tr(w)
    wn = w / sum(w)
    risk = sqrt(wn' * cov_tr * wn)
    return risk == 0 ? Inf : -(dot(wn, mu_tr) / risk)
end

res_train = Optim.optimize(neg_sharpe_tr, zeros(n_tr), ones(n_tr),
                     ones(n_tr) / n_tr, Fminbox(LBFGS()),
                     Optim.Options(iterations=300))
w_opt = res_train.minimizer / sum(res_train.minimizer)
w_eq = ones(n_tr) / n_tr

test_reduced = test[:, top_idx]
oos_sharpe(w, d) = (r = d * w; mean(r) / std(r))
println("Optimized OOS Sharpe:    ", round(oos_sharpe(w_opt, test_reduced), digits=4))
println("Equal-weight OOS Sharpe: ", round(oos_sharpe(w_eq, test_reduced), digits=4))

val_opt = exp.(cumsum(test_reduced * w_opt))
val_eq = exp.(cumsum(test_reduced * w_eq))
plot([scatter(x=ret_df.Date[split+1:end], y=val_opt, mode="lines",
              name="Optimized ($1 start)", line=attr(width=2, color="green")),
      scatter(x=ret_df.Date[split+1:end], y=val_eq, mode="lines",
              name="Equal weight ($1 start)",
              line=attr(width=2, color="grey", dash="dash"))],
     Layout(title="Out-of-sample: optimized vs equal-weight",
            xaxis_title="Date", yaxis_title="Portfolio value",
            width=1000, height=600, template="plotly_white"))